# 🛠 Feature Engineering

## Objective

The purpose of this notebook is to create meaningful features that enhance data analysis and improve the performance of future machine learning models.

These engineered features capture temporal patterns, geographical demand, and operational characteristics of emergency calls.

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries Imported Successfully")

✅ Libraries Imported Successfully


In [2]:
clean_df = pd.read_csv("../data/processed/911_preprocessed.csv")

print("✅ Preprocessed Dataset Loaded Successfully")

✅ Preprocessed Dataset Loaded Successfully


In [3]:
clean_df.head()

,lat,lng,desc,zip,title,timeStamp,twp,addr,e,Year,Month,Day,Hour,DayOfWeek,Weekend,TimeOfDay,Category,CategoryEncoded,Latitude_Scaled,Longitude_Scaled
0,40.297876,-75.581294,REINDEER CT & DEAD END; NEW HANOVER; Station ...,19525.0,EMS: BACK PAINS/INJURY,2015-12-10 17:10:52,NEW HANOVER,REINDEER CT & DEAD END,1,2015,12,10,17,Thursday,False,Afternoon,EMS,0,0.633100,-0.168057
1,40.258061,-75.264680,BRIAR PATH & WHITEMARSH LN; HATFIELD TOWNSHIP...,19446.0,EMS: DIABETIC EMERGENCY,2015-12-10 17:29:21,HATFIELD TOWNSHIP,BRIAR PATH & WHITEMARSH LN,1,2015,12,10,17,Thursday,False,Afternoon,EMS,0,0.452678,0.021171
2,40.121182,-75.351975,HAWS AVE; NORRISTOWN; 2015-12-10 @ 14:39:21-St...,19401.0,Fire: GAS-ODOR/LEAK,2015-12-10 14:39:21,NORRISTOWN,HAWS AVE,1,2015,12,10,14,Thursday,False,Afternoon,Fire,1,-0.167597,-0.031002
3,40.116153,-75.343513,AIRY ST & SWEDE ST; NORRISTOWN; Station 308A;...,19401.0,EMS: CARDIAC EMERGENCY,2015-12-10 16:47:36,NORRISTOWN,AIRY ST & SWEDE ST,1,2015,12,10,16,Thursday,False,Afternoon,EMS,0,-0.190386,-0.025944
4,40.251492,-75.603350,CHERRYWOOD CT & DEAD END; LOWER POTTSGROVE; S...,Unknown,EMS: DIZZINESS,2015-12-10 16:56:52,LOWER POTTSGROVE,CHERRYWOOD CT & DEAD END,1,2015,12,10,16,Thursday,False,Afternoon,EMS,0,0.422909,-0.181239


## 🚑 Peak Hour Feature

Peak emergency hours are defined as:

- 7 AM – 10 AM
- 4 PM – 8 PM

These periods usually experience higher emergency demand.

In [4]:
clean_df["PeakHour"] = clean_df["Hour"].between(7,10) | clean_df["Hour"].between(16,20)

clean_df[["Hour","PeakHour"]].head()

,Hour,PeakHour
0,17,True
1,17,True
2,14,False
3,16,True
4,16,True


In [5]:
clean_df["BusinessHours"] = clean_df["Hour"].between(9,17)

clean_df[["Hour","BusinessHours"]].head()

,Hour,BusinessHours
0,17,True
1,17,True
2,14,True
3,16,True
4,16,True


In [6]:
clean_df["RushHour"] = clean_df["Hour"].isin([7,8,9,16,17,18])

clean_df[["Hour","RushHour"]].head()

,Hour,RushHour
0,17,True
1,17,True
2,14,False
3,16,True
4,16,True


In [7]:
clean_df["NightEmergency"] = clean_df["Hour"].between(0,5)

clean_df[["Hour","NightEmergency"]].head()

,Hour,NightEmergency
0,17,False
1,17,False
2,14,False
3,16,False
4,16,False


In [8]:
def get_season(month):
    if month in [12,1,2]:
        return "Winter"
    elif month in [3,4,5]:
        return "Spring"
    elif month in [6,7,8]:
        return "Summer"
    else:
        return "Autumn"

clean_df["Season"] = clean_df["Month"].apply(get_season)

clean_df[["Month","Season"]].head()

,Month,Season
0,12,Winter
1,12,Winter
2,12,Winter
3,12,Winter
4,12,Winter


In [9]:
month_map = {
1:"January",
2:"February",
3:"March",
4:"April",
5:"May",
6:"June",
7:"July",
8:"August",
9:"September",
10:"October",
11:"November",
12:"December"
}

clean_df["MonthName"] = clean_df["Month"].map(month_map)

clean_df[["Month","MonthName"]].head()

,Month,MonthName
0,12,December
1,12,December
2,12,December
3,12,December
4,12,December


In [12]:
clean_df["Quarter"] = clean_df["timeStamp"].dt.quarter

clean_df[["Month","Quarter"]].head()

AttributeError: Can only use .dt accessor with datetimelike values

In [11]:
top_zip = clean_df["zip"].value_counts().head(10).index

clean_df["HighDemandZIP"] = clean_df["zip"].isin(top_zip)

clean_df[["zip","HighDemandZIP"]].head()

,zip,HighDemandZIP
0,19525.0,False
1,19446.0,True
2,19401.0,True
3,19401.0,True
4,Unknown,True


In [13]:
top_townships = clean_df["twp"].value_counts().head(10).index

clean_df["HighDemandTownship"] = clean_df["twp"].isin(top_townships)

clean_df[["twp","HighDemandTownship"]].head()

,twp,HighDemandTownship
0,NEW HANOVER,False
1,HATFIELD TOWNSHIP,False
2,NORRISTOWN,True
3,NORRISTOWN,True
4,LOWER POTTSGROVE,False


In [14]:
priority = {
"EMS":3,
"Fire":2,
"Traffic":1
}

clean_df["PriorityScore"] = clean_df["Category"].map(priority)

clean_df[["Category","PriorityScore"]].head()

,Category,PriorityScore
0,EMS,3
1,EMS,3
2,Fire,2
3,EMS,3
4,EMS,3


In [15]:
def density(hour):
    if hour <= 6:
        return "Low"
    elif hour <= 12:
        return "Medium"
    elif hour <= 18:
        return "High"
    else:
        return "Medium"

clean_df["CallDensity"] = clean_df["Hour"].apply(density)

clean_df[["Hour","CallDensity"]].head()

,Hour,CallDensity
0,17,High
1,17,High
2,14,High
3,16,High
4,16,High


In [16]:
new_features = [
"PeakHour",
"BusinessHours",
"RushHour",
"NightEmergency",
"Season",
"MonthName",
"Quarter",
"HighDemandTownship",
"HighDemandZIP",
"PriorityScore",
"CallDensity"
]

pd.DataFrame({
    "Feature Created": new_features,
    "Status": ["Completed"] * len(new_features)
})

,Feature Created,Status
0,PeakHour,Completed
1,BusinessHours,Completed
2,RushHour,Completed
3,NightEmergency,Completed
4,Season,Completed
5,MonthName,Completed
6,Quarter,Completed
7,HighDemandTownship,Completed
8,HighDemandZIP,Completed
9,PriorityScore,Completed


In [17]:
clean_df.to_csv(
    "../data/processed/911_feature_engineered.csv",
    index=False
)

print("✅ Feature Engineered Dataset Saved Successfully")

✅ Feature Engineered Dataset Saved Successfully


# ✅ Conclusion

Feature engineering transformed the preprocessed emergency response dataset into a richer analytical dataset.

New temporal, geographical, and operational features were created to better represent emergency response behavior.

These features improve interpretability and provide a strong foundation for predictive analytics and machine learning applications.